# High-Level Distributions

This notebook demonstrates composite distributions built from basic ones:

- `indep` - product of independent distributions
- `mixture` - mixture (weighted combination) of distributions
- `transformed` - apply bijective transformations

These allow building complex distributions from simple components.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import probjax.stats as stats
from probjax.stats import indep, mixture, transformed

key = jax.random.PRNGKey(0)

## Independent Distribution (`indep`)

`indep` creates a product distribution where each dimension is independent.
The most common pattern is to pass a batched distribution and reinterpret
the batch dimensions as event dimensions.

In [ ]:
# Independent Normals with different parameters via a batched distribution
key, subkey = jax.random.split(key)

base = stats.norm(loc=jnp.array([-2., 0., 3.]), scale=jnp.array([0.5, 1., 0.8]))
p = indep(base, reinterpreted_batch_ndims=1)

samples = p.rvs(subkey, shape=(5000,))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.hist(samples[:, i], bins=30, density=True, alpha=0.7)
    ax.set_title(f'Dimension {i+1}')
    ax.set_xlabel('x')
plt.tight_layout()
plt.show()

print(f"Batch shape: {p.batch_shape}")
print(f"Event shape: {p.event_shape}")
print(f"Mean: {p.mean()}")
print(f"Var:  {p.var()}")

In [ ]:
# Can also mix different distribution families
key, subkey = jax.random.split(key)

p2 = indep(
    stats.norm(loc=0., scale=1.),
    stats.expon(scale=2.),
    stats.beta(a=2., b=5.)
)

samples2 = p2.rvs(subkey, shape=(5000,))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(samples2[0::3], bins=30, density=True, alpha=0.7)
axes[0].set_title('Normal(0, 1)')
axes[1].hist(samples2[1::3], bins=30, density=True, alpha=0.7)
axes[1].set_title('Exponential(2)')
axes[2].hist(samples2[2::3], bins=30, density=True, alpha=0.7)
axes[2].set_title('Beta(2, 5)')
for ax in axes:
    ax.set_xlabel('x')
plt.tight_layout()
plt.show()

## Mixture Distribution (`mixture`)

`mixture(weights, components)` creates a weighted combination of distributions.
The `weights` array gives the probability of each component.

In [ ]:
# Gaussian Mixture Model (GMM)
key, subkey = jax.random.split(key)

components = [
    stats.norm(loc=-3., scale=0.5),
    stats.norm(loc=0., scale=1.0),
    stats.norm(loc=4., scale=0.7)
]
weights = jnp.array([0.3, 0.5, 0.2])

p = mixture(weights, components)

samples = p.rvs(subkey, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Mixture samples')

# Plot individual components
xs = jnp.linspace(-6, 7, 200)
for i, (comp, w) in enumerate(zip(components, weights)):
    plt.plot(xs, w * jnp.exp(comp.logpdf(xs)), '--', label=f'Component {i+1} (w={w:.1f})')

plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', linewidth=2, label='Mixture PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Gaussian Mixture Model')
plt.legend()
plt.show()

print(f"Mean: {p.mean():.3f}")
print(f"Var:  {p.var():.3f}")
print(f"Mode: {p.mode():.3f}")

In [ ]:
# 2D Mixture of Multivariate Normals
key, subkey = jax.random.split(key)

components_2d = [
    stats.multivariate_normal(
        loc=jnp.array([-2., -2.]),
        cov=jnp.array([[0.5, 0.], [0., 0.5]])
    ),
    stats.multivariate_normal(
        loc=jnp.array([2., 2.]),
        cov=jnp.array([[0.8, 0.3], [0.3, 0.8]])
    ),
    stats.multivariate_normal(
        loc=jnp.array([0., 3.]),
        cov=jnp.array([[0.3, 0.], [0., 0.3]])
    )
]
weights_2d = jnp.array([0.4, 0.4, 0.2])

p2d = mixture(weights_2d, components_2d)

samples_2d = p2d.rvs(subkey, shape=(5000,))

plt.figure(figsize=(8, 6))
plt.scatter(samples_2d[:, 0], samples_2d[:, 1], alpha=0.3, s=5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('2D Gaussian Mixture')
plt.axis('equal')
plt.show()

print(f"Mean: {p2d.mean()}")
print(f"Var shape: {p2d.var().shape}")

## Fitting Mixtures with EM

`mixture.fit` runs the Expectation-Maximization (EM) algorithm to estimate
both the component parameters and the mixing weights from data.

In [ ]:
key, subkey = jax.random.split(key)

# Ground-truth mixture
true_comps = [
    stats.norm(loc=-1.0, scale=0.5),
    stats.norm(loc=2.0, scale=0.8)
]
true_weights = jnp.array([0.35, 0.65])
true_mix = mixture(true_weights, true_comps)

# Generate data
key, subkey = jax.random.split(key)
data = true_mix.rvs(subkey, shape=(2000,))

plt.hist(data, bins=50, density=True, alpha=0.6, label='Observed data')
plt.title('Data from a 2-component Gaussian Mixture')
plt.legend()
plt.show()

# Fit with EM (initialize from rough guesses)
init_comps = [stats.norm(loc=0.0, scale=1.0), stats.norm(loc=1.0, scale=1.0)]
key, subkey = jax.random.split(key)
fitted_weights, fitted_comps = mixture.fit(
    data, init_comps, max_iter=100, tol=1e-4, rng_key=subkey
)

print("Fitted weights:", fitted_weights)
for i, comp in enumerate(fitted_comps):
    print(f"  Component {i}: mean={comp.mean():.3f}, std={comp.std():.3f}")

## Transformed Distribution (`transformed`)

`transformed(base_dist, bijector)` applies a bijective function to a base
distribution. The change-of-variables formula is handled automatically,
so `logpdf` and `pdf` are computed correctly.

Note: `mean()` and `var()` are not available for transformed distributions;
use sample estimates instead.

In [ ]:
# Log-normal distribution via exp transform
key, subkey = jax.random.split(key)

p = transformed(stats.norm(loc=0., scale=1.), jnp.exp)

samples = p.rvs(subkey, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Samples')

xs = jnp.linspace(0.1, 5, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='LogNormal PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Log-Normal (exp of Normal)')
plt.legend()
plt.show()

print(f"Sample mean: {jnp.mean(samples):.3f}")
print(f"Sample var:  {jnp.var(samples):.3f}")

In [ ]:
# Affine transformation: Shift and scale
key, subkey = jax.random.split(key)

base = stats.norm(loc=0., scale=1.)
shift = 5.0
scale = 2.0

p = transformed(base, lambda x: scale * x + shift)

samples = p.rvs(subkey, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Transformed samples')

xs = jnp.linspace(-2, 12, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title(f'Affine Transform: Normal -> Normal({shift}, {scale})')
plt.legend()
plt.show()

print(f"Sample mean: {jnp.mean(samples):.3f} (expected: {shift})")
print(f"Sample std:  {jnp.std(samples):.3f} (expected: {scale})")

In [ ]:
# Sigmoid transform to create a bounded distribution
key, subkey = jax.random.split(key)

base = stats.norm(loc=0., scale=2.)

# Transform through sigmoid: maps R -> (0, 1)
p = transformed(base, jax.nn.sigmoid)

samples = p.rvs(subkey, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Samples')

xs = jnp.linspace(0.01, 0.99, 100)
plt.plot(xs, jnp.exp(p.logpdf(xs)), 'r-', label='Sigmoid-Normal PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Sigmoid of Normal(0, 2)')
plt.legend()
plt.show()

print(f"All samples in (0, 1): {bool(jnp.all((samples > 0) & (samples < 1)))}")
print(f"Sample mean: {jnp.mean(samples):.3f}")

## Composition: Complex Distributions

Combine `mixture` and `transformed` for more complex models.

In [ ]:
# Mixture of transformed distributions
key, subkey = jax.random.split(key)

# Component 1: Log-normal (exp of Normal)
comp1 = transformed(stats.norm(loc=0., scale=0.5), jnp.exp)

# Component 2: Shifted exponential
comp2 = transformed(stats.expon(scale=1.), lambda x: x + 2.0)

# Component 3: Scaled beta
comp3 = transformed(stats.beta(a=2., b=2.), lambda x: 5.0 * x)

p_complex = mixture(
    jnp.array([0.3, 0.4, 0.3]),
    [comp1, comp2, comp3]
)

samples = p_complex.rvs(subkey, shape=(10000,))

plt.hist(samples, bins=50, density=True, alpha=0.7, label='Mixture samples')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Mixture of Transformed Distributions')
plt.legend()
plt.show()

print(f"Sample mean: {jnp.mean(samples):.3f}")
print(f"Sample var:  {jnp.var(samples):.3f}")

## Summary

High-level distributions compose basic ones:

| Class | Purpose | Example |
|-------|---------|---------|
| `indep` | Product distribution | IID normals |
| `mixture` | Weighted combination | GMM |
| `transformed` | Bijective transform | Log-normal |

These enable:
- **Modular modeling**: Build complex distributions from simple components
- **Automatic inference**: Transformations preserve probability (change of variables)
- **Flexible sampling**: All support `rvs()` with the same interface

For variational inference and probabilistic modeling, these primitives are essential building blocks.